In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit(
        "ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run "
        "jupyter lab)"
    )


DATA = data_dir()


# Exercise — Module 3: Regression

**Dataset (โปรดอ่านก่อนใช้งาน):**
- *NIST StRD "Pontius"* — Pontius, P., NIST. *Load Cell Calibration*. NIST StRD, https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml (public domain) — `datasets/Pontius.dat`
- *Combined Cycle Power Plant* — Tüfekci, P. & Kaya, H. (2014). UCI ML Repository. https://doi.org/10.24432/C5002N (CC BY 4.0) — `datasets/ccpp_power_plant.csv`

สองแบบฝึกหัด: (A) เช็กสมการสอบเทียบกับค่า certified ของ NIST, (B) เลือกโมเดล regression ที่ดีที่สุดสำหรับ CCPP อย่างซื่อสัตย์

---

**เวลาที่คาดหวังสำหรับแบบฝึกหัดนี้:** ~30 นาที (ทำในห้องช่วง lab — เฉลยเปิดเผยหลังจบช่วง)

ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)

In [2]:
import polars as pl  # noqa: E402
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans",
    "Noto Sans Thai",
    "Leelawadee UI",
    "Tahoma",
    "Thonburi",
]

pont = pl.DataFrame(
    np.loadtxt(DATA / "Pontius.dat", skiprows=60, max_rows=40),
    schema={"y": pl.Float64, "x": pl.Float64},
    orient="row",
)
ccpp = pl.read_csv(DATA / "ccpp_power_plant.csv")


## ข้อ A1 — พอดี quadratic แล้วเทียบ NIST certified

1. fit quadratic model ให้ได้สัมประสิทธิ์ B0, B1, B2 ในหน่วยดิบของ x
2. เทียบกับค่า certified: B0 = 0.673566e-3, B1 = 0.732059e-6, B2 = -0.316082e-14
3. รายงาน R² และ RMSE

⚠️ **กับดักตัวเลข:** `x` ~ 10⁶ ทำให้ `x²` ~ 10¹² — `LinearRegression` กับ `PolynomialFeatures` แบบดิบจะให้สัมประสิทธิ์เพี้ยน (ลองก่อนก็ได้ จะเห็นต่างจาก NIST มาก) ให้ใช้ `np.polyfit(x, y, 2)` ซึ่ง scale คอลัมน์ภายในเอง — คืนค่าเรียงจากดีกรีสูงไปต่ำ: `[B2, B1, B0]`

TODO

In [3]:
from sklearn.linear_model import LinearRegression  # noqa: E402, F401
from sklearn.preprocessing import PolynomialFeatures  # noqa: E402, F401
from sklearn.metrics import r2_score, mean_squared_error  # noqa: E402, F401

Xq = ...
quad = ...
b0, b1, b2 = ...
print(f"คุณ  : B0={b0:.6g}, B1={b1:.6g}, B2={b2:.6g}")
print("NIST : B0=0.000673566, B1=0.000000732059, B2=-3.16082e-15")
r2 = ...
rmse = ...
print(f"R² = {r2:.6f}, RMSE = {rmse:.4g}")


TypeError: cannot unpack non-iterable ellipsis object

**คำใบ้:** `PolynomialFeatures(degree=2, include_bias=False).fit_transform(pont.select("x"))` → คอลัมน์ `x, x²` แล้ว fit ตรง ๆ

## ข้อ A2 — residual ของสองโมเดล

พล็อต residual ต่อ x ของ (1) linear fit และ (2) quadratic fit — แถวสองแถบ ใช้ `plt.subplots(2, 1, figsize=(8, 5), sharex=True)`
แล้วตอบใน markdown: เทอมไหนทำให้รูปทรง U หายไป

TODO: สองแถบ residual plot

คำตอบข้อ A2:

## ข้อ A3 — ช่วงสอบเทียบและ extrapolation

พอดี quadratic เฉพาะจุดที่ `x <= 2_250_000` (22 จุดแรก) แล้วทำนายที่จุด x ที่เหลือ 18 จุด
รายงาน: RMSE ภายในช่วงเทรน vs RMSE ในช่วง extrapolate — และพล็อตเส้นทำนายเต็มช่วงพร้อมจุดข้อมูลจริง

⚠️ กับดักเดียวกับข้อ A1: scale x ด้วยสถิติของ train subset ก่อนพอดี (เช่น z = (x − mean)/std แล้ว fit เทอม z, z²)

TODO

In [4]:
mask_in = pont["x"] <= 2_250_000
train_p = pont.filter(mask_in)
test_p = pont.filter(~mask_in)

poly_low = ...
rmse_in = ...
rmse_out = ...
print(f"RMSE in-range = {rmse_in:.4g} | RMSE extrapolated = {rmse_out:.4g}")


TypeError: unsupported format string passed to ellipsis.__format__

## ข้อ B1 — โมเดลเทียบโมเดลบน CCPP

สร้างสาม pipeline บน CCPP (split 80/20, random_state=42):
1. `StandardScaler → LinearRegression`
2. `StandardScaler → PolynomialFeatures(2) → Ridge(alpha=10)`
3. `StandardScaler → PolynomialFeatures(2) → Lasso(alpha=0.1)`

รายงาน test R² ของแต่ละตัว และจำนวนสัมประสิทธิ์ที่ lasso กดเป็นศูนย์

TODO

In [5]:
from sklearn.pipeline import make_pipeline  # noqa: E402, F401
from sklearn.preprocessing import StandardScaler  # noqa: E402, F401
from sklearn.model_selection import train_test_split  # noqa: E402, F401
from sklearn.linear_model import Ridge, Lasso  # noqa: E402, F401

X = ccpp.select("AT", "V", "AP", "RH")
y = ccpp["PE"]
X_train, X_test, y_train, y_test = ...

r2s = {}
for name, model in [...]:
    ...


TypeError: cannot unpack non-iterable ellipsis object

**คำใบ้:** นับสัมประสิทธิ์ศูนย์: `(np.abs(lasso.named_steps["lasso"].coef_) < 1e-10).sum()`

## ข้อ B2 — เลือก alpha ด้วย CV

ใช้ `GridSearchCV` (cv=5, scoring="r2") กับ pipeline `StandardScaler → Ridge` ค้น alpha ใน [0.01, 0.1, 1, 10, 100] รายงาน best alpha, best CV score, และ test R²

TODO

In [6]:
from sklearn.model_selection import GridSearchCV  # noqa: E402, F401

search = ...
print(
    f"best alpha = {search.best_params_}, CV R² = {
        search.best_score_:.4f}, test R² = {r2:.4f}"
)


AttributeError: 'ellipsis' object has no attribute 'best_params_'

## ข้อ B3 — โจทย์คิด (ตอบใน markdown)

ถ้าเพื่อนร่วมชั้นอวดว่าโมเดลของเขาได้ test R² = 0.9999 บน CCPP — คำถามอะไรที่คุณควรถามก่อนเชื่อ? (นึกถึง: split, leakage, extrapolation, และว่า feature ไหนได้อะไรมา)

คำตอบข้อ B3: